# Phase 2 - TOON vs JSON vs YAML Benchmark
In this project, we will compare structured data formats for LLM/RAG workflows using a real PocketPolls dataset.

We will compare:

- Pretty JSON
- Minified JSON
- YAML
- TOON

The goal is to measure token efficiency, file size, and later RAG answer quality.

Note - The PocketPolls dataset does not contain any PII or Sensitive User data.

In [1]:
#tiktoken is OpenAI’s tokenizer library; it lets us count tokens before sending text to a model, which is useful for cost and context-window planning.
!pip install tiktoken pyyaml pandas

In [2]:
#Upload all the 4 Input Files in format JSON Minified, YAML, Pretty JSON and TOON

from google.colab import files

uploaded = files.upload()

Saving pocketpolls_JSON_Minified.json to pocketpolls_JSON_Minified.json
Saving pocketpolls_JSON_Pretty.json to pocketpolls_JSON_Pretty.json
Saving pocketpolls_TOON.toon to pocketpolls_TOON.toon
Saving pocketpolls_YAML.yaml to pocketpolls_YAML.yaml


# Step 3 - Verify Uploaded Files

Before processing the files, we verify that all benchmark datasets were uploaded successfully.

Expected files:

- pocketpolls_JSON_Minified.json
- pocketpolls_YAML.yaml
- pocketpolls_TOON.toon


In [3]:
# ============================================================
# STEP 3
#
# Verify uploaded files.
#
# This helps catch upload mistakes early and confirms
# that all benchmark formats are available.
# ============================================================

import os

print("Uploaded Files")
print("-" * 50)

for filename in uploaded.keys():

    size_kb = os.path.getsize(filename) / 1024

    print(f"{filename}")
    print(f"Size: {size_kb:.2f} KB")
    print()

Uploaded Files
--------------------------------------------------
pocketpolls_JSON_Minified.json
Size: 115.40 KB

pocketpolls_JSON_Pretty.json
Size: 174.19 KB

pocketpolls_TOON.toon
Size: 135.93 KB

pocketpolls_YAML.yaml
Size: 129.84 KB



# Step 4 - Load Raw Content

At this stage we intentionally treat each format as raw text.

The goal is to compare how different formats behave during retrieval.

No parsing or normalization will occur yet.

In [4]:
# ============================================================
# STEP 4
#
# Load all files as raw text.
#
# We intentionally keep the formats untouched because
# we want to compare how each representation performs
# inside a RAG pipeline.
# ============================================================

files_data = {} # A dictionary that will hold the entire content of each file string to maintain a raw copy to be used later for comparison

for filename in uploaded.keys():

    with open(filename, "r", encoding="utf-8") as f:

        files_data[filename] = f.read()

print("Files loaded successfully.")

Files loaded successfully.


# Step 5 - Baseline Dataset Statistics

Before chunking, we measure:

- Characters
- Words
- Tokens

These values establish a baseline for later comparisons.

In [5]:
# ============================================================
# STEP 5
#
# Baseline Statistics
#
# Measure:
#
# - Characters
# - Words
# - Tokens
#
# These metrics help explain downstream retrieval
# behavior and embedding costs.
# ============================================================

import tiktoken
import pandas as pd

encoding = tiktoken.get_encoding("cl100k_base")

stats = []

for filename, content in files_data.items():

    stats.append({
        "Format": filename,
        "Characters": len(content),
        "Words": len(content.split()),
        "Tokens": len(encoding.encode(content))
    })

stats_df = pd.DataFrame(stats)

stats_df

,Format,Characters,Words,Tokens
0,pocketpolls_JSON_Minified.json,116155,10227,32061
1,pocketpolls_JSON_Pretty.json,176358,16996,42465
2,pocketpolls_TOON.toon,137178,14518,35334
3,pocketpolls_YAML.yaml,130964,15788,36615


Above in Step 5, you can clearly see that the number of Characters, Words and Tokens are less in JSON Minified - which suggests that it is already a great format in our case - likely bacause the pretty JSON had a lot of spaces which got removed. TOON on the other hand comes 2nd - reason why TOON is token optimized.

# Step 6 - Chunking Experiment

In a RAG pipeline, the source document is usually split into smaller chunks before embedding.

This step applies the same chunking strategy to each format:

- JSON Minified
- YAML
- TOON

The goal is to compare how each representation behaves when prepared for retrieval.

We will measure:

- Number of chunks
- Average tokens per chunk
- Minimum chunk size
- Maximum chunk size

In [6]:
# ============================================================
# STEP 6
#
# Chunking Experiment
#
# Goal:
#
# Split each file into smaller text chunks using the same
# token-based chunking method.
#
# We will perform token-based chunking for this comparison test.
# Why token-based chunking?
#
# LLMs process tokens, not characters.
# Token-based chunking gives us more consistent control over
# how much content each chunk contains.
#
# Chunk size:
#
# 500 tokens per chunk
#
# Overlap:
#
# 50 tokens
#
# Overlap helps preserve context between neighboring chunks.
# ============================================================

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

def chunk_text_by_tokens(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split text into overlapping token-based chunks.

    Parameters:
    text:
        The raw text we want to split.

    chunk_size:
        Maximum number of tokens per chunk.

    overlap:
        Number of tokens repeated between consecutive chunks.

    Returns:
        A list of text chunks.
    """

    tokens = encoding.encode(text)

    chunks = []

    start = 0

    while start < len(tokens):

        end = start + chunk_size

        chunk_tokens = tokens[start:end]

        chunk_text = encoding.decode(chunk_tokens)

        chunks.append(chunk_text)

        start = end - overlap

    return chunks

# Engineering Findings Log

This section captures observations, tradeoffs, and quality considerations discovered throughout the study.

As I am learning new things via experimentation and hit n trials, I am capturing them and my goal for this case study has evolved to document "decision-making frameworks" rather than one size fits all.

# Engineering Findings

## Engineering Finding #1: Token Efficiency ≠ Retrieval Efficiency

Minified JSON was the most token-efficient representation, but that does not automatically imply superior retrieval performance.

Retrieval quality depends on:

- Chunking strategy
- Embedding quality
- Semantic boundaries
- Query characteristics

**Key Insight:** Storage efficiency and retrieval effectiveness should be evaluated independently.

---

## Engineering Finding #2: Optimal Chunk Size is Workload-Dependent

There is no universal chunk size that performs best across all workloads.

The appropriate chunk size depends on:

- Document structure
- Query patterns
- Retrieval objectives
- Context window constraints

**Key Insight:** Chunk size should be selected based on the retrieval task rather than a fixed rule.

---

## Engineering Finding #3: Natural Boundaries May Outperform Arbitrary Boundaries

For the PocketPolls dataset, individual cards represent self-contained units of meaning.

Future experiments should compare:

- Pollcard-based chunking specific to pocketpolls.com

to determine whether semantic chunking improves retrieval effectiveness.

**Key Insight:** Semantic boundaries may produce more meaningful retrieval units than fixed-size chunks.

---

## Engineering Finding #4: Chunk Size Must Balance Retrieval and Context Constraints

A chunk should be:

- Large enough to preserve meaning
- Small enough to enable precise retrieval
- Small enough that multiple retrieved chunks can fit comfortably within the target model's context window

The optimal chunk size is therefore a function of:

- Data structure
- Query patterns
- Retrieval strategy
- Model context window

**Key Insight:** Chunk size optimization is a multi-variable engineering tradeoff.

---

## Engineering Finding #5: Chunk Size Does Not Directly Determine Hallucination Rates

Chunk size does not directly determine hallucination rates. However, chunking decisions influence:

- Retrieval precision
- Context completeness
- Context dilution

These factors can affect the likelihood of unsupported or incorrect responses.

A more accurate statement is:

> Chunking decisions indirectly influence hallucination risk through their impact on retrieval quality.

**Key Insight:** The relationship between chunk size and hallucinations is indirect and mediated by retrieval performance.

In [7]:
# ============================================================
# STEP 6A
#
# Generate chunks for each format.
#
# We exclude Pretty JSON because:
#
# - Same semantics as Minified JSON
# - Differences are only formatting
# - Already evaluated in Phase 1
#
# ============================================================

all_chunks = {}

for filename, content in files_data.items():

    chunks = chunk_text_by_tokens(content)

    all_chunks[filename] = chunks

    print(f"{filename}: {len(chunks)} chunks")

pocketpolls_JSON_Minified.json: 72 chunks
pocketpolls_JSON_Pretty.json: 95 chunks
pocketpolls_TOON.toon: 79 chunks
pocketpolls_YAML.yaml: 82 chunks


## Engineering Finding #6: Dataset Representation Influences Chunk Count

The same information does not necessarily produce the same number of chunks.

In this study, all formats contained:

- The same 28 cards from pocketpolls.com
- The same metadata
- The same statistics
- The same generation prompts

However, the chunking process produced different chunk counts because each representation consumes tokens differently.

As a result, chunk count is not purely a property of the dataset. It is a property of:

- The dataset representation
- The chunking strategy

**Key Insight:** Identical information can produce different retrieval structures depending on how that information is encoded.

---

## Engineering Finding #7A: Duplication Overhead is a Retrieval Tradeoff

**Duplication overhead** is the additional information volume introduced by data representation, chunking strategy, or retrieval architecture beyond the original dataset.

Common sources include:

- Chunk overlap
- Repeated metadata
- Hierarchical chunking
- Multiple vector representations
- Redundant source content

A certain amount of duplication is often necessary to preserve context and improve retrieval quality. However, excessive duplication can increase:

- Storage costs
- Embedding costs
- Retrieval complexity
- Contextual noise

without proportional gains in answer accuracy.

### QA Perspective

The goal is **not** to minimize duplication overhead to zero.

The goal is to identify the **minimum duplication necessary to achieve acceptable retrieval quality**.

**Key Insight:** Duplication should be treated as an engineering investment whose cost must be justified by measurable retrieval improvements.

## Engineering Finding #7B: Chunk Count and Total Chunk Tokens Influence Different System Costs

Two related but distinct metrics emerged during the experiment:

### Number of Chunks - is the total number of individual chunks produced after chuinking process

The total number of chunks affects:

- Retrieval search space
- Embedding count
- Storage requirements

As chunk count increases, the retrieval system must evaluate a larger set of candidate chunks, and more embeddings must be generated and stored.

### Total Chunk Tokens (Including Overlap) - The sum of the tokens contained across all chunks after chunking, including any duplicated tokens introduced by overlap.

The total number of tokens across all chunks, including duplicated tokens introduced through overlap, affects:

- Embedding cost
- Storage cost
- Indexing cost

Even if two systems contain the same number of chunks, the system with more total chunk tokens will generally require more processing and storage resources.

### Key Insight

Chunk count and total chunk tokens measure different aspects of retrieval complexity.

- **Chunk Count** primarily influences retrieval search space and embedding quantity.
- **Total Chunk Tokens** primarily influences computational cost, storage consumption, and indexing overhead.

Both metrics should be evaluated when assessing the efficiency of a RAG architecture.

In [8]:
# ============================================================
# STEP 6B
#
# Chunking Metrics
#
# Goal:
#
# Measure:
#
# - Original tokens
# - Number of chunks
# - Total chunk tokens
# - Duplication overhead
#
# Why?
#
# Chunk overlap improves context preservation but creates
# duplicated content that increases embedding and storage costs.
#
# ============================================================

chunk_stats = []

for filename, chunks in all_chunks.items():

    original_tokens = len(
        encoding.encode(files_data[filename])
    )

    chunk_token_counts = [
        len(encoding.encode(chunk))
        for chunk in chunks
    ]

    total_chunk_tokens = sum(chunk_token_counts)

    duplication_overhead = (
        total_chunk_tokens - original_tokens
    )

    duplication_pct = round(
        (duplication_overhead / original_tokens) * 100,
        2
    )

    chunk_stats.append({
        "Format": filename,
        "Original Tokens": original_tokens,
        "Chunks": len(chunks),
        "Chunked Tokens": total_chunk_tokens,
        "Duplication Overhead": duplication_overhead,
        "Duplication %": duplication_pct
    })

chunk_stats_df = pd.DataFrame(chunk_stats)

chunk_stats_df

,Format,Original Tokens,Chunks,Chunked Tokens,Duplication Overhead,Duplication %
0,pocketpolls_JSON_Minified.json,32061,72,35609,3548,11.07
1,pocketpolls_JSON_Pretty.json,42465,95,47163,4698,11.06
2,pocketpolls_TOON.toon,35334,79,39233,3899,11.03
3,pocketpolls_YAML.yaml,36615,82,40663,4048,11.06


# Phase 4 - Embeddings and Retrieval Preparation

## Objective

The chunking phase transformed each dataset representation into retrievable units.

The next step is to convert those chunks into vector embeddings.

Embeddings are numerical representations of text that allow similarity search to be performed based on meaning rather than exact keyword matching.

This phase focuses on building the retrieval layer of the benchmark.

To ensure a fair comparison:

- JSON Minified
- JSON Pretty
- TOON
- YAML

will all use:

- The same chunking strategy
- The same embedding model
- The same retrieval process

This isolates the effect of data representation and allows us to evaluate whether different formats influence retrieval behavior and downstream LLM response quality.

## Why Embeddings Matter

Large Language Models do not search raw files directly.

Instead, Retrieval-Augmented Generation (RAG) systems typically:

1. Split source data into chunks.
2. Convert chunks into embeddings.
3. Store embeddings in a vector index.
4. Retrieve the most relevant chunks for a query.
5. Provide retrieved context to an LLM.

The quality of retrieval directly impacts the quality of the final answer.

## Benchmark Goal

The primary objective of this study is not to determine the best chunking strategy.

Instead, the goal is to evaluate whether different representations of the same information:

- JSON
- TOON
- YAML

lead to differences in retrieval effectiveness and LLM response quality when all other variables remain constant.

Token-based chunking was intentionally selected as a controlled baseline so that data representation remains the primary variable under evaluation.

In [9]:
# ============================================================
# STEP 7
#
# Install the OpenAI Python SDK.
#
# We will use it to create embeddings for each chunk.
# ============================================================

!pip install openai

In [10]:
# ============================================================
# STEP 8
#
# Load OpenAI API Key from Colab Secrets.
#
# The key is stored securely in Colab and is not
# visible in notebook code cells.
#
# ============================================================

from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get(
    "OPENAI_API_KEY"
)

print("OpenAI API key loaded successfully.")

OpenAI API key loaded successfully.


In [11]:
#STEP 8A

from openai import OpenAI

client = OpenAI()

print("OpenAI client initialized.")

OpenAI client initialized.


In [12]:
# ============================================================
# STEP 9A
#
# Connectivity Test
#
# Before generating hundreds of embeddings,
# verify that authentication is working.
#
# ============================================================

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="PocketPolls RAG benchmark test"
)

print("Embedding API connection successful.")
print(
    "Embedding dimension:",
    len(response.data[0].embedding)
)

Embedding API connection successful.
Embedding dimension: 1536


## Step 10 - Create Embedding Helper

To keep the notebook maintainable, we encapsulate embedding generation inside a reusable function.

This improves:

- readability
- reusability
- maintainability

and makes it easier to swap embedding models in future experiments.

For this benchmark we use:

- text-embedding-3-small

The same embedding model will be used across all formats to ensure a fair comparison.

In [13]:
# ============================================================
# STEP 10
#
# Embedding Helper Function
#
# Creates embeddings using OpenAI's
# text-embedding-3-small model.
#
# Using a helper function centralizes
# embedding logic and improves maintainability.
#
# Converts text into embedding vectors
#
# ============================================================

EMBEDDING_MODEL = "text-embedding-3-small"

def get_embedding(text):

    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )

    return response.data[0].embedding

## Step 11 - Generate Chunk Embeddings

Each chunk is converted into a vector embedding.

Embeddings capture semantic meaning and enable similarity-based retrieval.

For this benchmark:

- Each format is embedded independently.
- The same embedding model is used throughout.
- The same chunking strategy is used throughout.

This ensures that data representation remains the primary variable under evaluation.

In [14]:
# ============================================================
# STEP 11
#
# Generate embeddings for all chunks created in the previous step.
#
# We maintain separate embedding collections
# for each format.
#
# This allows us to compare retrieval quality
# later while keeping all other variables fixed.
#
# ============================================================

embeddings_by_format = {}

for filename, chunks in all_chunks.items():

    print("=" * 70)
    print(f"Processing {filename}")
    print("=" * 70)

    format_embeddings = []

    for idx, chunk in enumerate(chunks):

        embedding = get_embedding(chunk)

        format_embeddings.append(embedding)

        if (idx + 1) % 10 == 0:
            print(
                f"Completed {idx + 1} / {len(chunks)} chunks"
            )

    embeddings_by_format[filename] = format_embeddings

    print(
        f"Finished {filename} "
        f"({len(format_embeddings)} embeddings)"
    )
    print()

print("All embeddings generated successfully.")

Processing pocketpolls_JSON_Minified.json
Completed 10 / 72 chunks
Completed 20 / 72 chunks
Completed 30 / 72 chunks
Completed 40 / 72 chunks
Completed 50 / 72 chunks
Completed 60 / 72 chunks
Completed 70 / 72 chunks
Finished pocketpolls_JSON_Minified.json (72 embeddings)

Processing pocketpolls_JSON_Pretty.json
Completed 10 / 95 chunks
Completed 20 / 95 chunks
Completed 30 / 95 chunks
Completed 40 / 95 chunks
Completed 50 / 95 chunks
Completed 60 / 95 chunks
Completed 70 / 95 chunks
Completed 80 / 95 chunks
Completed 90 / 95 chunks
Finished pocketpolls_JSON_Pretty.json (95 embeddings)

Processing pocketpolls_TOON.toon
Completed 10 / 79 chunks
Completed 20 / 79 chunks
Completed 30 / 79 chunks
Completed 40 / 79 chunks
Completed 50 / 79 chunks
Completed 60 / 79 chunks
Completed 70 / 79 chunks
Finished pocketpolls_TOON.toon (79 embeddings)

Processing pocketpolls_YAML.yaml
Completed 10 / 82 chunks
Completed 20 / 82 chunks
Completed 30 / 82 chunks
Completed 40 / 82 chunks
Completed 50 / 82

# Step 12 - Create Vector Indexes

Now that every chunk has been converted into an embedding, we need a way to search those embeddings.

An embedding is a long list of numbers that represents the meaning of a text chunk.

However, having embeddings alone is not enough.

We need a search structure that can answer:

> “Which chunks are most similar to this user question?”

That search structure is called a vector index.

In this notebook, we use FAISS - Since we dont have a dedicated DB like PineCone or Postgres-pgvector
FAISS is a library designed for fast similarity search over vectors.

For this benchmark, we create a separate FAISS index for each format:

- JSON Minified
- JSON Pretty
- TOON
- YAML

This allows us to test retrieval separately for each representation while keeping the retrieval process consistent.

In [15]:
# ============================================================
# STEP 12
#
# Install FAISS.
#
# FAISS is a vector search library.
#
# In normal search, we search text using keywords.
#
# In vector search, we search by meaning.
#
# Example:
#
# A user may ask:
#   "Which poll showed disagreement?"
#
# The exact word "disagreement" may not exist in the dataset.
#
# But a vector search system can still find related chunks
# containing words like:
#
#   "divide"
#   "split"
#   "polarized"
#
# This is why embeddings + vector search are useful for RAG.
# ============================================================

!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 67.0 MB/s eta 0:00:00


## Step 12A - Build One Vector Index Per Format

Each format gets its own vector index.

This is important for fairness.

We do not mix JSON, YAML, and TOON chunks into the same index because that would make it difficult to evaluate each format independently.

Instead, we create:

- one index for JSON Minified chunks
- one index for JSON Pretty chunks
- one index for TOON chunks
- one index for YAML chunks

Later, when we ask a question, we will search each index separately and compare the results.

In [16]:
# ============================================================
# STEP 12A
#
# Build FAISS indexes.
#
# What we have right now:
#
# embeddings_by_format
#
# This is a dictionary.
#
# Example:
#
# {
#   "pocketpolls_JSON_Minified.json": [embedding_1, embedding_2, ...],
#   "pocketpolls_TOON.toon": [embedding_1, embedding_2, ...]
# }
#
# Each embedding is a long list of numbers.
#
# FAISS expects these embeddings to be stored in a NumPy array.
#
# So for each format, we:
#
# 1. Convert the embeddings into a NumPy matrix.
# 2. Detect the embedding dimension.
# 3. Create a FAISS index.
# 4. Add the embeddings to the index.
# 5. Store the index for later retrieval.
#
# ============================================================

import faiss
import numpy as np

indexes_by_format = {}

for filename, embeddings in embeddings_by_format.items():

    # Convert the list of embeddings into a NumPy matrix.
    #
    # FAISS requires float32 format.
    #
    # Shape example:
    #
    # 72 chunks x 1536 embedding dimensions
    embedding_matrix = np.array(
        embeddings,
        dtype="float32"
    )

    # The embedding dimension is the number of numeric values
    # in each embedding vector.
    #
    # For example, text-embedding-3-small commonly returns
    # 1536-dimensional vectors.
    dimension = embedding_matrix.shape[1]

    # Create a FAISS index using inner product similarity.
    #
    # Because OpenAI embeddings are already suitable for
    # similarity comparison, this allows us to rank chunks
    # based on semantic closeness to the query.
    index = faiss.IndexFlatIP(dimension)

    # Add all chunk embeddings for this format into the index.
    index.add(embedding_matrix)

    # Store the index so we can search it later.
    indexes_by_format[filename] = index

    print(
        f"{filename}: FAISS index created "
        f"with {index.ntotal} vectors "
        f"and {dimension} dimensions"
    )

pocketpolls_JSON_Minified.json: FAISS index created with 72 vectors and 1536 dimensions
pocketpolls_JSON_Pretty.json: FAISS index created with 95 vectors and 1536 dimensions
pocketpolls_TOON.toon: FAISS index created with 79 vectors and 1536 dimensions
pocketpolls_YAML.yaml: FAISS index created with 82 vectors and 1536 dimensions


## Engineering Finding #10: Chunk Count Directly Determines Vector Count

Chunk count directly determines the number of vectors stored in the retrieval index.

Since each chunk generates exactly one embedding:

```text
More Chunks
    ↓
More Embeddings
    ↓
Larger Vector Index
    ↓
Higher Vector Storage Cost
```

Every additional chunk requires:

- One additional embedding to be generated
- One additional vector to be stored
- One additional searchable entry in the retrieval index

As a result, chunk count has a direct impact on the size of the retrieval index and the number of searchable vectors within a RAG system.



###Two Types of COSTS
### Storage Cost

Storage cost consists of two separate components:

#### Vector Storage

Primarily determined by chunk count.

```text
More Chunks
    ↓
More Embeddings
    ↓
More Stored Vectors
```

Since each chunk produces exactly one embedding:

- More chunks = more vectors
- More vectors = larger vector index

#### Text Storage

Primarily determined by total chunk tokens.

```text
Larger Chunks / More Overlap
    ↓
More Total Chunk Tokens
    ↓
More Text Stored
```

Even if two systems contain the same number of chunks, the system with more total chunk tokens will require more text storage.

### Indexing Cost

Indexing cost is influenced by two factors:

1. Number of vectors (chunk count)
2. Embedding generation workload (total chunk tokens)

For example:

| Scenario | Chunks | Avg Chunk Size | Total Tokens |
|-----------|---------|---------|---------|
| A | 100 | 500 | 50,000 |
| B | 100 | 1,000 | 100,000 |

Both scenarios generate:

```text
100 vectors
```

However, Scenario B requires approximately:

- 2× the text to process
- 2× the embedding workload
- 2× the token-related indexing effort

despite having the same number of vectors.

### Key Insight

Chunk count and total chunk tokens influence different aspects of system cost:

| Metric | Primary Impact |
|----------|----------|
| Chunk Count | Vector count, vector storage, retrieval search space |
| Total Chunk Tokens | Embedding cost, text storage, indexing workload |

A complete evaluation of RAG efficiency should consider both metrics rather than focusing on chunk count alone.

# What Have We Built So Far?

### Current Architecture

```text
PocketPolls Dataset
        ↓
JSON / TOON / YAML
        ↓
Token Chunking
        ↓
OpenAI Embeddings
        ↓
FAISS Vector Index
```

### Component Breakdown

#### 1. PocketPolls Dataset

The original source dataset containing:

- Poll cards
- Statistics
- Metadata
- Generation prompts

#### 2. JSON / TOON / YAML

The same dataset represented in multiple formats to evaluate:

- Storage efficiency
- Token efficiency
- Retrieval behavior

#### 3. Token Chunking

Documents are divided into smaller retrieval units.

The chunking strategy determines:

- Chunk count
- Total chunk tokens
- Retrieval granularity

#### 4. OpenAI Embeddings

Each chunk is converted into a vector representation using:

```python
text-embedding-3-small
```

Result:

```text
Chunk
    ↓
Embedding
    ↓
Vector
```

#### 5. FAISS Vector Index

The generated embeddings are stored in a FAISS index to enable semantic similarity search.

Result:

```text
User Question
        ↓
Question Embedding
        ↓
Similarity Search
        ↓
Top Matching Chunks
```

### Current System Status

✅ Dataset Loaded

✅ Multiple Representations Generated

✅ Documents Chunked

✅ Embeddings Generated

✅ FAISS Index Built

**Outcome:** A complete Retrieval-Augmented Generation (RAG) retrieval pipeline capable of performing semantic search across JSON, TOON, and YAML representations of the PocketPolls dataset.



What remains is:

User Question
        ↓
Question Embedding
        ↓
FAISS Retrieval
        ↓
Retrieved Chunks
        ↓
LLM Answer

# The Next Really Interesting Step

Up until now, this study has focused primarily on measuring the efficiency characteristics of different data representations.

Metrics evaluated so far include:

- Token efficiency
- Chunk count
- Duplication overhead
- Vector count

These measurements help us understand the engineering cost of each representation, but they do not tell us whether one representation actually retrieves better information.

---

## The Next Phase: Retrieval Quality

The next stage of the study shifts focus from efficiency to effectiveness.

Key questions include:

- Does JSON retrieve more relevant chunks than TOON or YAML?
- Do different representations produce different similarity rankings?
- Does one representation consistently retrieve the correct chunk more often?
- How do chunking strategies influence retrieval accuracy?

At this stage, retrieval quality becomes the primary evaluation criterion.

---

## The Final Phase: LLM Response Quality

Ultimately, retrieval exists to support question answering.

The final objective of the study is therefore not:

- Smallest file size
- Lowest token count
- Fewest chunks

The final objective is:

> Producing the most accurate and reliable answers.

Key questions include:

- Do all representations generate the same answer?
- Does one representation provide more complete context?
- Does one representation reduce hallucinations?
- Does one representation improve answer accuracy or consistency?

---

## Key Insight

Engineering efficiency metrics are important, but they are ultimately supporting metrics.

The true success criterion is:

```text
Representation
        ↓
Retrieval Quality
        ↓
LLM Response Quality
        ↓
User Value
```

A representation that is slightly larger or less token-efficient may still be superior if it consistently produces better retrieval results and higher-quality answers.

# Some Engineering Findings We've Already Discovered

## Finding #1: Token Efficiency and Retrieval Efficiency Are Different

JSON Minified was the most token-efficient representation in the study.

However, token efficiency alone does not determine retrieval quality.

A format that uses fewer tokens is not necessarily the format that retrieves the most relevant information.

**Key Insight:** Storage efficiency and retrieval effectiveness should be evaluated independently.

---

## Finding #2: Chunk Count Is Influenced by Representation

The same information can produce different chunk counts depending on how it is represented.

| Format | Chunks |
|----------|----------|
| JSON Minified | 72 |
| TOON | 79 |
| YAML | 82 |
| JSON Pretty | 95 |

Although all formats contained identical information, differences in formatting and tokenization resulted in different chunk counts.

**Key Insight:** Chunk count is a property of both the dataset and its representation.

---

## Finding #3: Overlap Creates Intentional Duplication

Chunk overlap intentionally duplicates content across neighboring chunks.

Without overlap:

```text
Meaning may be split across chunk boundaries.
```

With overlap:

```text
More context preserved.
More storage consumed.
```

This additional duplication is deliberate and is designed to improve retrieval quality.

**Key Insight:** Overlap is an engineering tradeoff, not a defect.

---

## Finding #4: Embeddings Are Not Knowledge

A common misconception is:

```text
Embedding = Stored Knowledge
```

In reality, embeddings are mathematical representations that make content searchable.

```text
Knowledge
    ↓
Chunk
    ↓
Embedding
```

The actual knowledge remains within the original chunk text.

**Key Insight:** Embeddings store semantic relationships, not the underlying knowledge itself.

---

## Finding #5: The LLM Is Not the Knowledge Store

Another common misconception is that the LLM contains all knowledge used by a RAG system.

In reality:

```text
Source Data
    ↓
Chunks
    ↓
Embeddings
    ↓
Vector Store
```

The knowledge resides within the external retrieval system.

At runtime:

```text
User Question
    ↓
Retrieve Chunks
    ↓
LLM Receives Context
    ↓
Generate Answer
```

**Key Insight:** In RAG architectures, knowledge lives outside the LLM.

---

## Finding #6: Most Document Embeddings Are Generated Once

Document embeddings are typically generated during indexing rather than at query time.

```text
Document Embedding
        =
Indexing Time
```

```text
Question Embedding
        =
Query Time
```

If document embeddings were regenerated for every query, retrieval costs would become prohibitively expensive.

**Key Insight:** Embedding generation is usually a one-time indexing operation.

---

## Finding #7: Data Freshness Requires Re-indexing

RAG systems do not automatically stay synchronized with their source data.

When source content changes:

```text
Changed Content
        ↓
Re-chunk
        ↓
Re-embed
        ↓
Update Vector Store
```

Without re-indexing, retrieval results may become outdated.

**Key Insight:** Maintaining data freshness requires an explicit re-indexing strategy.

Phase 5 - Retrieval Benchmark

Until now we've measured:

Storage
Token Efficiency
Chunk Counts
Embedding Counts
Index Size

Now we want to measure:

Can the system find the right information?

# Step 13 - Find the Most Relevant Chunks

We now have:

✓ Chunks

✓ Embeddings

✓ Searchable indexes

The next step is simple:

Given a question, can we find the most relevant parts of the dataset?

This is the retrieval step of RAG.

At this stage we are not asking the LLM to answer anything.

We are only checking whether the system can find the right information.

In [17]:
# ============================================================
# STEP 13 - Retrieval Function - To be called later
#
# Workflow:
# 1. Embed the user question
# 2. Search the FAISS index
# 3. Retrieve top matching chunks
# 4. Return chunk text + similarity scores
#
# Note:
# This stage evaluates retrieval quality only.
# No LLM is involved yet.
# ============================================================

def retrieve_chunks(question, filename, top_k=3):

    # Convert question into embedding
    question_embedding = get_embedding(question)

    # FAISS requires a float32 matrix
    query_vector = np.array(
        [question_embedding],
        dtype="float32"
    )

    # Search vector index
    scores, indices = indexes_by_format[
        filename
    ].search(
        query_vector,
        top_k
    )

    # Build readable results
    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        results.append({

            "score": float(score),

            "chunk_id": int(idx),

            "chunk": all_chunks[
                filename
            ][idx]

        })

    return results

In [22]:
# ============================================================
# STEP 14 - Retrieval Sanity Check
#
# Purpose:
# Run the same question against every format and compare:
# - Similarity scores
# - Retrieved chunks
# - Retrieval consistency
#
# Formats:
# JSON Minified, JSON Pretty, TOON, YAML
# ============================================================

# **************************************TEST QUESTION *****************************************
test_question = (
    "Will AI reduce the importance of teachers?"
)
#********************************************************************************************

# Search all formats
for filename in indexes_by_format.keys():

    print("\n")
    print("=" * 80)
    print(filename)
    print("=" * 80)

    # Retrieve top 3 matching chunks
    results = retrieve_chunks(
        question=test_question,
        filename=filename,
        top_k=3
    )

    # Display results
    for rank, result in enumerate(
        results,
        start=1
    ):

        print(f"\nResult {rank}")

        print(
            "Similarity Score:",
            round(result["score"], 4)
        )

        print(
            "Chunk ID:",
            result["chunk_id"]
        )

        print("\nChunk Preview:")
        print("Chunk Preview: [Redacted for public release]")

        print("\n" + "-" * 80)



pocketpolls_JSON_Minified.json

Result 1
Similarity Score: 0.5522
Chunk ID: 4

Chunk Preview:
Chunk Preview: [Redacted for public release]

--------------------------------------------------------------------------------

Result 2
Similarity Score: 0.5149
Chunk ID: 15

Chunk Preview:
Chunk Preview: [Redacted for public release]

--------------------------------------------------------------------------------

Result 3
Similarity Score: 0.3873
Chunk ID: 14

Chunk Preview:
Chunk Preview: [Redacted for public release]

--------------------------------------------------------------------------------


pocketpolls_JSON_Pretty.json

Result 1
Similarity Score: 0.4344
Chunk ID: 5

Chunk Preview:
Chunk Preview: [Redacted for public release]

--------------------------------------------------------------------------------

Result 2
Similarity Score: 0.4186
Chunk ID: 6

Chunk Preview:
Chunk Preview: [Redacted for public release]

-----------------------------------------------------------------

Engineering Finding #13

A very nice takeaway for the case study:

Retrieval and generation should be evaluated independently.

A RAG system can fail before the LLM is ever invoked. Poor retrieval often leads to poor answers, even when using a powerful model. Evaluating retrieval quality separately makes root-cause analysis significantly easier and aligns with Quality Engineering principles.

# Step 15 - Build the Answer Generation Pipeline

So far we have completed:

✓ Chunking

✓ Embedding Generation

✓ Vector Search

✓ Retrieval Validation

At this point we know the system can retrieve relevant chunks.

The next step is to prepare the answer generation pipeline.

In this step we create a reusable function that will:

1. Accept a user question.
2. Accept retrieved chunks.
3. Send the question and retrieved context to GPT.
4. Return an answer.

No answers are generated yet.

We are only building the infrastructure that will be used in the next step.

The function created here will be executed in Step 16.

In [23]:
# ============================================================
# STEP 15 - LLM Answer Generation
#
# Purpose:
# Convert retrieved chunks into a grounded answer.
#
# Workflow:
# Question
#    ↓
# Retrieved Chunks
#    ↓
# LLM
#    ↓
# Final Answer
#
# ============================================================

ANSWER_MODEL = "gpt-4o-mini"

def generate_answer_from_chunks(question, retrieved_chunks):

    # Combine retrieved chunks into one context block
    context = "\n\n---\n\n".join(
        [
            chunk["chunk"]
            for chunk in retrieved_chunks
        ]
    )

    # Ground the model in retrieved context
    prompt = f"""
You are answering questions using only the retrieved context below.

Rules:
1. Use only the provided context.
2. Do not use outside knowledge.
3. Do not invent facts.
4. If the context is insufficient, say: "The retrieved context does not contain enough information."
5. Keep the answer concise and specific.
6. Do not mention the file format unless the question asks about it.

Question:
{question}

Retrieved Context:
{context}

Answer:
"""

    # Generate answer
    response = client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

# Step 16 - Generate Answers Using Retrieved Context

We now execute the answer generation pipeline.

For each format:

- JSON Minified
- JSON Pretty
- TOON
- YAML

we will:

1. Retrieve the most relevant chunks.
2. Send those chunks to GPT.
3. Generate an answer.

Everything remains constant except the source representation.

This allows us to evaluate whether different representations of the same information influence the final RAG answer.

This is the first step in the notebook where GPT is asked to generate answers.

In [24]:
# ============================================================
# STEP 16 - Execute the RAG Benchmark
#
# Purpose:
# Compare end-to-end RAG performance across formats.
#
# Workflow:
# Question
#    ↓
# Retrieval
#    ↓
# Retrieved Chunks
#    ↓
# GPT Answer
#    ↓
# Benchmark Results
#
# ============================================================

# Benchmark question
question = (
    "Will AI reduce the importance of teachers?"
)

# Number of chunks to retrieve
top_k = 5

# Store results
rag_results = []

# Process each format
for filename in indexes_by_format.keys():

    print("\n" + "=" * 80)
    print(f"Processing {filename}")
    print("=" * 80)

    # Retrieve relevant chunks
    retrieved_chunks = retrieve_chunks(
        question=question,
        filename=filename,
        top_k=top_k
    )

    # Generate answer from retrieved chunks
    answer = generate_answer_from_chunks(
        question=question,
        retrieved_chunks=retrieved_chunks
    )

    # Save benchmark result
    rag_results.append({

        "Format": filename,

        "Top Similarity Score":
            round(
                retrieved_chunks[0]["score"],
                4
            ),

        "Top Chunk ID":
            retrieved_chunks[0]["chunk_id"],

        "Generated Answer":
            answer

    })


# ============================================================
# Results Summary Table
# ============================================================

rag_results_df = pd.DataFrame(
    rag_results
)

display(rag_results_df)


# ============================================================
# Full Answers (No Truncation)
# ============================================================

print("\n")
print("=" * 80)
print("FULL GENERATED ANSWERS")
print("=" * 80)

for result in rag_results:

    print("\n" + "=" * 80)

    print(
        f"Format: "
        f"{result['Format']}"
    )

    print(
        f"Top Similarity Score: "
        f"{result['Top Similarity Score']}"
    )

    print(
        f"Top Chunk ID: "
        f"{result['Top Chunk ID']}"
    )

    print("\nGenerated Answer:\n")

    print(
        result["Generated Answer"]
    )

    print("\n")


Processing pocketpolls_JSON_Minified.json

Processing pocketpolls_JSON_Pretty.json

Processing pocketpolls_TOON.toon

Processing pocketpolls_YAML.yaml


,Format,Top Similarity Score,Top Chunk ID,Generated Answer
0,pocketpolls_JSON_Minified.json,0.5522,4,The retrieved context indicates that 100% of v...
1,pocketpolls_JSON_Pretty.json,0.4344,5,The retrieved context indicates that 100% of v...
2,pocketpolls_TOON.toon,0.4885,16,AI will support teachers and will not replace ...
3,pocketpolls_YAML.yaml,0.5783,5,AI will support teachers and will not replace ...




FULL GENERATED ANSWERS

Format: pocketpolls_JSON_Minified.json
Top Similarity Score: 0.5522
Top Chunk ID: 4

Generated Answer:

The retrieved context indicates that 100% of voters believe "AI will support teachers, will not replace them," suggesting that AI will not reduce the importance of teachers.



Format: pocketpolls_JSON_Pretty.json
Top Similarity Score: 0.4344
Top Chunk ID: 5

Generated Answer:

The retrieved context indicates that 100% of voters believe "AI will support teachers, will not replace them," suggesting that AI will not reduce the importance of teachers.



Format: pocketpolls_TOON.toon
Top Similarity Score: 0.4885
Top Chunk ID: 16

Generated Answer:

AI will support teachers and will not replace them, according to 100% of the voters in the context provided.



Format: pocketpolls_YAML.yaml
Top Similarity Score: 0.5783
Top Chunk ID: 5

Generated Answer:

AI will support teachers and will not replace them, according to 100% of the voters in the context provided.




In [25]:
# ============================================================
# STEP 17
#
# Display full generated answers.
#
# The dataframe preview truncates long text, so we print
# each answer separately for proper review.
# ============================================================

for row in rag_results:

    print("=" * 80)
    print(row["Format"])
    print("=" * 80)

    print("Top Similarity Score:", row["Top Similarity Score"])
    print("Top Chunk ID:", row["Top Chunk ID"])

    print("\nGenerated Answer:")
    print(row["Generated Answer"])

    print("\n")

pocketpolls_JSON_Minified.json
Top Similarity Score: 0.5522
Top Chunk ID: 4

Generated Answer:
The retrieved context indicates that 100% of voters believe "AI will support teachers, will not replace them," suggesting that AI will not reduce the importance of teachers.


pocketpolls_JSON_Pretty.json
Top Similarity Score: 0.4344
Top Chunk ID: 5

Generated Answer:
The retrieved context indicates that 100% of voters believe "AI will support teachers, will not replace them," suggesting that AI will not reduce the importance of teachers.


pocketpolls_TOON.toon
Top Similarity Score: 0.4885
Top Chunk ID: 16

Generated Answer:
AI will support teachers and will not replace them, according to 100% of the voters in the context provided.


pocketpolls_YAML.yaml
Top Similarity Score: 0.5783
Top Chunk ID: 5

Generated Answer:
AI will support teachers and will not replace them, according to 100% of the voters in the context provided.




## Engineering Finding #14: Similarity Score Is a Retrieval Metric, Not an Answer-Quality Metric

Similarity scores measure how closely a retrieved chunk matches the query in embedding space.

They do **not** directly measure answer quality.

For example:

```text
Similarity Score = 0.55
```

may produce a perfectly correct answer if the retrieved chunk contains all required information.

Conversely:

```text
Similarity Score = 0.75
```

may still produce a poor answer if important context is missing from the retrieved results.

### Example

```text
Higher Similarity Score
        ≠
Better Answer
```

because answer quality depends on factors beyond retrieval similarity, including:

- Context completeness
- Context relevance
- Retrieval ranking
- LLM reasoning

### Key Insight

Similarity score should be interpreted as a **retrieval metric**, not an **answer-quality metric**.

A strong RAG evaluation must assess both:

1. Retrieval Quality
2. Answer Quality

rather than relying solely on similarity scores.

# Step 18 - Define Benchmark Questions

To evaluate answer quality fairly, a single question is insufficient.

A benchmark question set is created covering:

- factual retrieval
- opinion splits
- demographic analysis
- correlations
- trends
- uncertainty / harder reasoning

Each file format will be evaluated using the same questions.

In [26]:
benchmark_questions = [
    {
        "category": "Direct factual",
        "question": "Will AI reduce the importance of teachers?"
    },
    {
        "category": "Direct factual",
        "question": "Who should set the rules for AI?"
    },
    {
        "category": "Direct factual",
        "question": "Who built the internet age?"
    },
    {
        "category": "Opinion split",
        "question": "Which topic had the strongest disagreement?"
    },
    {
        "category": "Trend",
        "question": "Which topic showed changing sentiment over time?"
    },
    {
        "category": "Hard query",
        "question": "What were people most uncertain about?"
    }
]

len(benchmark_questions)

6

# Step 19 - Run Benchmark Across All Formats

Each benchmark question is tested against each representation:

- JSON Minified
- TOON
- YAML

For every combination, the pipeline will:

Question → Retrieve Context → Generate Answer

The results are stored in a dataframe for comparison.

In [27]:
# ============================================================
# STEP 19 - Run Benchmark Question Set Across All Formats
# ============================================================

top_k = 5

benchmark_results = []

for item in benchmark_questions:

    category = item["category"]
    question = item["question"]

    for filename in indexes_by_format.keys():

        retrieved_chunks = retrieve_chunks(
            question=question,
            filename=filename,
            top_k=top_k
        )

        answer = generate_answer_from_chunks(
            question=question,
            retrieved_chunks=retrieved_chunks
        )

        benchmark_results.append({
            "Category": category,
            "Question": question,
            "Format": filename,
            "Top Similarity Score": round(
                retrieved_chunks[0]["score"],
                4
            ),
            "Top Chunk ID": retrieved_chunks[0]["chunk_id"],
            "Generated Answer": answer
        })


benchmark_results_df = pd.DataFrame(
    benchmark_results
)

# ============================================================
# Results Summary
# ============================================================

display(
    benchmark_results_df[
        [
            "Category",
            "Question",
            "Format",
            "Top Similarity Score",
            "Top Chunk ID"
        ]
    ]
)

# ============================================================
# Full Answers (No Truncation)
# ============================================================

pd.set_option(
    "display.max_colwidth",
    None
)

display(
    benchmark_results_df[
        [
            "Category",
            "Question",
            "Format",
            "Generated Answer"
        ]
    ]
)

,Category,Question,Format,Top Similarity Score,Top Chunk ID
0,Direct factual,Will AI reduce the importance of teachers?,pocketpolls_JSON_Minified.json,0.5522,4
1,Direct factual,Will AI reduce the importance of teachers?,pocketpolls_JSON_Pretty.json,0.4344,5
2,Direct factual,Will AI reduce the importance of teachers?,pocketpolls_TOON.toon,0.4886,16
3,Direct factual,Will AI reduce the importance of teachers?,pocketpolls_YAML.yaml,0.5783,5
4,Direct factual,Who should set the rules for AI?,pocketpolls_JSON_Minified.json,0.4081,4
5,Direct factual,Who should set the rules for AI?,pocketpolls_JSON_Pretty.json,0.3538,6
6,Direct factual,Who should set the rules for AI?,pocketpolls_TOON.toon,0.4115,43
7,Direct factual,Who should set the rules for AI?,pocketpolls_YAML.yaml,0.4221,5
8,Direct factual,Who built the internet age?,pocketpolls_JSON_Minified.json,0.3527,41
9,Direct factual,Who built the internet age?,pocketpolls_JSON_Pretty.json,0.3045,51


,Category,Question,Format,Generated Answer
0,Direct factual,Will AI reduce the importance of teachers?,pocketpolls_JSON_Minified.json,"The retrieved context indicates that 100% of voters believe ""AI will support teachers, will not replace them,"" suggesting that AI will not reduce the importance of teachers."
1,Direct factual,Will AI reduce the importance of teachers?,pocketpolls_JSON_Pretty.json,"The retrieved context indicates that 100% of voters believe ""AI will support teachers, will not replace them,"" suggesting that AI will not reduce the importance of teachers."
2,Direct factual,Will AI reduce the importance of teachers?,pocketpolls_TOON.toon,"AI will support teachers and will not replace them, according to 100% of the voters in the retrieved context."
3,Direct factual,Will AI reduce the importance of teachers?,pocketpolls_YAML.yaml,"AI will support teachers and will not replace them, according to 100% of the voters in the context provided."
4,Direct factual,Who should set the rules for AI?,pocketpolls_JSON_Minified.json,"Independent Experts should set the rules for AI, as indicated by 40% of the votes."
5,Direct factual,Who should set the rules for AI?,pocketpolls_JSON_Pretty.json,"Independent experts should set the rules for AI, as indicated by 40% of the votes."
6,Direct factual,Who should set the rules for AI?,pocketpolls_TOON.toon,"The rules for AI should be set by Independent Experts, as indicated by the top option in the retrieved context."
7,Direct factual,Who should set the rules for AI?,pocketpolls_YAML.yaml,"The retrieved context indicates that 40% of voters believe ""Independent Experts"" should set the rules for AI."
8,Direct factual,Who built the internet age?,pocketpolls_JSON_Minified.json,"The retrieved context indicates that 58.8% of voters believe that ""American vision & Silicon Valley money"" built the internet age."
9,Direct factual,Who built the internet age?,pocketpolls_JSON_Pretty.json,"The retrieved context indicates that the top option for who built the internet age is ""🇺🇸 American vision & Silicon Valley money,"" with 58.8% of votes supporting this view."


# Engineering Findings

## Engineering Finding #1
JSON Minified was the most storage-efficient representation.

| Format | Size (Bytes) | Tokens |
|----------|----------:|----------:|
| JSON Minified | 118,167 | 32,061 |
| TOON | 139,190 | 35,334 |
| YAML | 132,961 | 36,615 |
| JSON Pretty | 178,370 | 42,465 |

JSON Minified reduced both file size and token count compared to all other formats.

---

## Engineering Finding #2
Human-readable formatting carries a significant token cost.

JSON Pretty contained the same information as JSON Minified but required approximately 32% more tokens due solely to whitespace and formatting.

---

## Engineering Finding #3
All formats preserved identical information content.

Validation confirmed that JSON Minified, JSON Pretty, TOON, and YAML contained the same dataset, card count, metadata, and poll information after sanitization.

---

## Engineering Finding #4
Format conversion can be performed without information loss.

The dataset was successfully converted between JSON, TOON, and YAML while preserving semantic meaning and structural integrity.

---

## Engineering Finding #5
Chunking and embedding generation were format-agnostic.

The same chunking strategy and embedding model were successfully applied across all representations.

---

## Engineering Finding #6
Retrieval quality varied across formats.

Similarity scores differed between formats even when answering the same question, indicating that representation can influence embedding and retrieval behavior.

---

## Engineering Finding #7
The highest similarity score did not always produce the best answer.

Examples from the benchmark showed that higher retrieval scores sometimes produced weaker answers, while lower-scoring chunks occasionally generated correct answers.

---

## Engineering Finding #8
Similarity score is a retrieval metric, not an answer-quality metric.

A chunk with a similarity score of 0.40–0.50 can still generate a correct answer, while a higher-scoring chunk may produce a weaker answer if important context is missing.

---

## Engineering Finding #9
The benchmark successfully demonstrated an end-to-end RAG workflow.

All formats were able to:

Question
→ Retrieval
→ Context Assembly
→ LLM Generation
→ Final Answer

This validated that the benchmark pipeline functioned correctly across all representations.

---

## Engineering Finding #10
Representation can influence retrieval outcomes.

For certain benchmark questions, one format retrieved sufficient context while another format returned insufficient context, suggesting that representation may affect retrieval behavior.

---

## Engineering Finding #11
Increasing retrieval depth from Top-3 to Top-5 did not materially change benchmark outcomes.

The answer-generation function combines all retrieved chunks before sending them to the LLM. Despite increasing retrieval depth, answer quality remained largely unchanged, indicating that benchmark outcomes were primarily driven by the highest-ranked retrieved chunks.

---

## Engineering Finding #12
The benchmark question suite provided a more meaningful evaluation than a single test question.

The benchmark included:

- Direct factual retrieval
- Opinion splits
- Trend identification
- Hard reasoning queries

This allowed evaluation of answer quality across multiple retrieval scenarios rather than relying on a single example.

---

## Engineering Finding #13
Harder analytical questions were more sensitive to retrieval quality.

Questions involving uncertainty, trends, and synthesis were more likely to produce unsupported or incomplete answers than direct factual retrieval questions.

---

## Engineering Finding #14
Direct factual questions produced highly consistent results across formats.

Questions such as:

- Will AI reduce the importance of teachers?
- Who should set the rules for AI?

were answered correctly by nearly all representations.

---

## Engineering Finding #15
Across the benchmark question set, all four representations produced broadly similar answer quality.

Manual evaluation produced the following scores:

| Format | Score |
|----------|----------:|
| JSON Pretty | 16 |
| JSON Minified | 15 |
| YAML | 14 |
| TOON | 14 |

The differences were small and insufficient to conclude that any representation consistently outperformed the others.

---

## Engineering Finding #16
Storage efficiency varied significantly across formats, but answer quality remained largely comparable.

The benchmark suggests that representation has a much larger impact on storage cost than on end-to-end RAG answer quality.

For this dataset:

- Storage efficiency differed substantially.
- Retrieval behavior showed minor variation.
- Final answer quality remained broadly similar.

## Engineering Finding #17
In this benchmark, the largest differences appeared in storage efficiency, while answer quality remained relatively stable across representations.

---

# Final Conclusion

For the PocketPolls AI Vibe Check dataset, JSON Minified was the most storage-efficient representation, achieving the lowest file size and token count.

However, benchmark testing showed that JSON Minified, JSON Pretty, TOON, and YAML produced broadly comparable answer quality within the RAG pipeline.

The results suggest that representation choice has a stronger impact on storage efficiency than on downstream answer quality, at least for this dataset, chunking strategy, embedding model, and retrieval configuration.

Future work could expand the benchmark question set, evaluate additional datasets, and test alternative chunking and retrieval strategies to determine whether these findings generalize across broader RAG workloads.